In [1]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/TwinCheck'
REPO = '/content/Group-Project'
OUT = f'{DRIVE}/results/F6'
CKPT = f'{DRIVE}/models/F5'
!rm -rf {REPO}
!git clone -q --depth 1 https://github.com/himaallu/Group-Project.git {REPO}
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
import io
import os
import pickle
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

os.chdir(REPO)
sys.path.insert(0, REPO)
from src.vision.detectors import build_model, eval_transform
from src.vision.gradcam import grad_cam, unnormalise
from src.vision.train_detector import metrics

SPLITS = ('train', 'validation', 'test')
ARCHS = ('resnet50', 'vit_b16')
NAMES = {'resnet50': 'ResNet-50', 'vit_b16': 'ViT-B/16'}
P = [f'p_ai_{a}' for a in ARCHS]
CACHE = f'{DRIVE}/data/processed/vision/genimage_debiased96_jpeg.pkl'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
os.makedirs(f'{OUT}/cases', exist_ok=True)

gen = pd.concat([pd.read_csv(f'data/processed/vision/genimage_{s}_manifest.csv') for s in SPLITS], ignore_index=True)
with open(CACHE, 'rb') as f:
    cache = pickle.load(f)
assert cache['md5'] == gen['md5'].tolist() and len(gen) == 13000
BLOBS = cache['jpeg']
assert gen.groupby(['split', 'generator']).size().to_dict() == {
    ('test', 'Stable Diffusion v1.4'): 1000, ('test', 'real'): 1000,
    ('train', 'BigGAN'): 5000, ('train', 'real'): 4000,
    ('validation', 'BigGAN'): 1000, ('validation', 'real'): 1000}
assert all(os.path.exists(f'{CKPT}/{a}_s42.pt') for a in ARCHS), 'Run F5 first'
assert os.path.exists('results/F5/cnn_vit.csv')
assert DEVICE == 'cuda', 'Runtime > Change runtime type > T4 GPU'
print('images:', len(gen), '| 96 px input from F3/F5 cache:', round(sum(map(len, BLOBS)) / 1e6, 1), 'MB | device:', torch.cuda.get_device_name(0))

images: 13000 | 96 px input from F3/F5 cache: 92.2 MB | device: Tesla T4


In [3]:
class Inputs(Dataset):
    def __init__(self, index):
        self.index, self.transform = list(index), eval_transform()

    def __len__(self):
        return len(self.index)

    def __getitem__(self, k):
        return self.transform(Image.open(io.BytesIO(BLOBS[self.index[k]])).convert('RGB'))


nets = {}
for arch in ARCHS:
    net = build_model(arch, pretrained=False).to(DEVICE, memory_format=torch.channels_last)
    net.load_state_dict(torch.load(f'{CKPT}/{arch}_s42.pt', map_location=DEVICE)['model'])
    nets[arch] = net.eval()

parts, check = [], []
for split in ('validation', 'test'):
    index = np.flatnonzero(gen['split'] == split)
    rows = gen.iloc[index].reset_index(drop=True)
    probs = {a: [] for a in ARCHS}
    with torch.inference_mode():
        for x in tqdm(DataLoader(Inputs(index), batch_size=64), desc=f'Scoring {split}'):
            x = x.to(DEVICE).contiguous(memory_format=torch.channels_last)
            for arch, net in nets.items():
                with torch.autocast('cuda', dtype=torch.float16):
                    probs[arch].append(torch.sigmoid(net(x).squeeze(1).float()).cpu().numpy())
    table = rows[['md5', 'split', 'generator', 'class_name', 'original_format', 'original_width', 'original_height', 'raw_export_path']].copy()
    table['gen_index'] = index
    y = (rows['class_name'] == 'fake').astype(int).to_numpy()
    for arch in ARCHS:
        table[f'p_ai_{arch}'] = np.concatenate(probs[arch])
        check.append({'model': arch, 'split': split, **metrics(table[f'p_ai_{arch}'].to_numpy(), y, rows)})
    parts.append(table)
pred = pd.concat(parts, ignore_index=True)
pred.drop(columns=['raw_export_path', 'gen_index']).to_csv(f'{OUT}/predictions.csv', index=False)

cols = ['accuracy', 'precision', 'recall', 'f1', 'auroc']
ref = pd.read_csv('results/F5/cnn_vit.csv').query("corruption == 'clean'")[['model', 'split'] + cols]
cmp = pd.DataFrame(check)[['model', 'split'] + cols].merge(ref, on=['model', 'split'], suffixes=('', '_f5'))
cmp['max_abs_diff'] = np.abs(cmp[cols].to_numpy() - cmp[[c + '_f5' for c in cols]].to_numpy()).max(axis=1)
cmp.to_csv(f'{OUT}/check_against_f5.csv', index=False)
print(cmp.round(4).to_string(index=False))
assert cmp['max_abs_diff'].max() < 0.002, 'The scores do not match results/F5/cnn_vit.csv'

Scoring validation:   0%|          | 0/32 [00:00<?, ?it/s]

Scoring test:   0%|          | 0/32 [00:00<?, ?it/s]

   model      split  accuracy  precision  recall     f1  auroc  accuracy_f5  precision_f5  recall_f5  f1_f5  auroc_f5  max_abs_diff
resnet50 validation    0.8425     0.9971   0.687 0.8135 0.9818       0.8425        0.9971      0.687 0.8135    0.9818           0.0
 vit_b16 validation    0.9915     0.9920   0.991 0.9915 0.9995       0.9915        0.9920      0.991 0.9915    0.9995           0.0
resnet50       test    0.5045     0.7647   0.013 0.0256 0.6099       0.5045        0.7647      0.013 0.0256    0.6099           0.0
 vit_b16       test    0.5550     0.8873   0.126 0.2207 0.7542       0.5550        0.8873      0.126 0.2207    0.7542           0.0


In [4]:
rows = []
for split in ('validation', 'test'):
    part = pred[pred['split'] == split]
    for (generator, cls), g in part.groupby(['generator', 'class_name']):
        for arch in ARCHS:
            wrong = (g[f'p_ai_{arch}'] >= 0.5) != (cls == 'fake')
            rows.append({'split': split, 'group': generator, 'model': arch, 'images': len(g), 'errors': int(wrong.sum()),
                         'error_rate': wrong.mean(), 'mean_p_ai': g[f'p_ai_{arch}'].mean()})
error_summary = pd.DataFrame(rows)
error_summary.to_csv(f'{OUT}/error_summary.csv', index=False)
print(error_summary.round(3).to_string(index=False))

test = pred[pred['split'] == 'test']
sd = test[test['class_name'] == 'fake']
real = test[test['class_name'] == 'real']
both_miss = sd[(sd[P] < 0.5).all(axis=1)]
vit_only = sd[(sd['p_ai_resnet50'] < 0.5) & (sd['p_ai_vit_b16'] >= 0.5)]
resnet_only = sd[(sd['p_ai_resnet50'] >= 0.5) & (sd['p_ai_vit_b16'] < 0.5)]
overlap = pd.DataFrame([{'group': 'Stable Diffusion v1.4 (test)', 'missed_by_both': len(both_miss), 'caught_by_vit_only': len(vit_only),
                         'caught_by_resnet_only': len(resnet_only), 'caught_by_both': len(sd) - len(both_miss) - len(vit_only) - len(resnet_only)}])
overlap.to_csv(f'{OUT}/error_overlap.csv', index=False)
print(overlap.to_string(index=False))

real_all = pred[pred['class_name'] == 'real'].copy()
real_all['short_side'] = real_all[['original_width', 'original_height']].min(axis=1)
real_all['size_group'] = pd.cut(real_all['short_side'], [0, 127, 223, 511, 10 ** 6], labels=['under 128 px', '128-223 px', '224-511 px', '512 px and over'])
by_size = pd.DataFrame([{'split': split, 'short_side_of_original': str(size), 'real_images': len(g),
                         **{f'false_positives_{a}': int((g[f'p_ai_{a}'] >= 0.5).sum()) for a in ARCHS},
                         **{f'fp_rate_{a}': float((g[f'p_ai_{a}'] >= 0.5).mean()) for a in ARCHS}}
                        for (split, size), g in real_all.groupby(['split', 'size_group'], observed=True)])
by_size.to_csv(f'{OUT}/real_errors_by_original_size.csv', index=False)
print(by_size.round(3).to_string(index=False))

gen['short_side'] = gen[['original_width', 'original_height']].min(axis=1)
audit = gen.groupby(['split', 'class_name', 'generator', 'original_format']).agg(
    images=('md5', 'size'), width_min=('original_width', 'min'), width_median=('original_width', 'median'), width_max=('original_width', 'max'),
    height_min=('original_height', 'min'), height_median=('original_height', 'median'), height_max=('original_height', 'max'),
    non_square=('original_width', lambda w: int((w.to_numpy() != gen.loc[w.index, 'original_height'].to_numpy()).sum()))).reset_index()
audit.to_csv(f'{OUT}/data_audit.csv', index=False)
print(audit.to_string(index=False))

     split                 group    model  images  errors  error_rate  mean_p_ai
validation                BigGAN resnet50    1000     313       0.313      0.642
validation                BigGAN  vit_b16    1000       9       0.009      0.986
validation                  real resnet50    1000       2       0.002      0.010
validation                  real  vit_b16    1000       8       0.008      0.012
      test Stable Diffusion v1.4 resnet50    1000     987       0.987      0.032
      test Stable Diffusion v1.4  vit_b16    1000     874       0.874      0.138
      test                  real resnet50    1000       4       0.004      0.013
      test                  real  vit_b16    1000      16       0.016      0.020
                       group  missed_by_both  caught_by_vit_only  caught_by_resnet_only  caught_by_both
Stable Diffusion v1.4 (test)             872                 115                      2              11
     split short_side_of_original  real_images  false_positives

In [5]:
assert len(both_miss) >= 4 and len(vit_only) >= 2 and len(real) >= 2
picks = [
    ('confident miss, both models', both_miss.loc[both_miss[P].mean(axis=1).nsmallest(2).index]),
    ('borderline miss, both models', both_miss.loc[both_miss[P].max(axis=1).nlargest(2).index]),
    ('caught by ViT-B/16 only', vit_only.nlargest(2, 'p_ai_vit_b16')),
    ('false positive on real', real.loc[real[P].max(axis=1).nlargest(2).index]),
]
cases = pd.concat([part.assign(case=name) for name, part in picks], ignore_index=True)
cases.insert(0, 'case_id', range(1, len(cases) + 1))
for arch in ARCHS:
    cases[f'correct_{arch}'] = (cases[f'p_ai_{arch}'] >= 0.5) == (cases['class_name'] == 'fake')

x = torch.stack([Inputs(cases['gen_index'])[i] for i in range(len(cases))]).to(DEVICE)
cams = {}
for arch, net in nets.items():
    cams[arch], cases[f'p_ai_gradcam_{arch}'] = grad_cam(net, arch, x)
cases.drop(columns=['gen_index']).to_csv(f'{OUT}/cases.csv', index=False)
for _, row in cases.iterrows():
    shutil.copy(f"{DRIVE}/{row['raw_export_path']}", f"{OUT}/cases/case{row['case_id']}_{row['md5']}{Path(row['raw_export_path']).suffix.lower()}")

fig, axes = plt.subplots(3, 8, figsize=(24, 10))
for i, row in cases.iterrows():
    axes[0, i].imshow(unnormalise(x[i]))
    axes[0, i].set_title(f"Case {row['case_id']}: {row['generator']}\n{row['case']}\noriginal {row['original_width']}x{row['original_height']} {row['original_format']}", fontsize=8)
    for r, arch in enumerate(ARCHS, start=1):
        axes[r, i].imshow(unnormalise(x[i]))
        axes[r, i].imshow(cams[arch][i], cmap='jet', alpha=0.4)
        axes[r, i].set_title(f"{NAMES[arch]}  P(AI) = {row[f'p_ai_{arch}']:.3f}", fontsize=8)
for ax in axes.flat:
    ax.axis('off')
fig.suptitle('Grad-CAM on eight test cases (96 px input, seed 42, threshold 0.5)')
fig.tight_layout()
fig.savefig(f'{OUT}/fig_f6_error_cases.png', dpi=200)
plt.show()
plt.close(fig)
print(cases[['case_id', 'case', 'generator', 'original_width', 'original_height', 'original_format'] + P + ['p_ai_gradcam_resnet50', 'p_ai_gradcam_vit_b16']].round(4).to_string(index=False))

 case_id                         case             generator  original_width  original_height original_format  p_ai_resnet50  p_ai_vit_b16  p_ai_gradcam_resnet50  p_ai_gradcam_vit_b16
       1  confident miss, both models Stable Diffusion v1.4             512              512             PNG         0.0000        0.0002                 0.0000                0.0002
       2  confident miss, both models Stable Diffusion v1.4             512              512             PNG         0.0000        0.0003                 0.0000                0.0003
       3 borderline miss, both models Stable Diffusion v1.4             512              512             PNG         0.0778        0.4910                 0.0776                0.4913
       4 borderline miss, both models Stable Diffusion v1.4             512              512             PNG         0.0012        0.4831                 0.0012                0.4831
       5      caught by ViT-B/16 only Stable Diffusion v1.4             512          

In [6]:
print(pd.read_csv(f'{OUT}/check_against_f5.csv')[['model', 'split', 'max_abs_diff']].to_string(index=False))
for f in sorted(os.listdir(OUT)):
    print(f)
print(sorted(os.listdir(f'{OUT}/cases')))

   model      split  max_abs_diff
resnet50 validation  0.000000e+00
 vit_b16 validation  0.000000e+00
resnet50       test  3.469447e-18
 vit_b16       test  5.551115e-17
cases
cases.csv
check_against_f5.csv
data_audit.csv
error_overlap.csv
error_summary.csv
fig_f6_error_cases.png
predictions.csv
real_errors_by_original_size.csv
['case1_5c83e32fb002e54aa96a425c939ba778.png', 'case2_7f30b25f7d3c16e5483b753e89a7ae9e.png', 'case3_a14a5fb92cf1b68e674a615a94ae998c.png', 'case4_0a9fd8645b2cdb8dc3779ac24b95ffcf.png', 'case5_21ee1990c8759b982081fe76ff7ad245.png', 'case6_986a0aadd962bf25b247dfc026cdc930.png', 'case7_6f22aaf51bffa9ddb3269ddbda4c0ba6.jpg', 'case8_f82f65cbc5d3540a390cc087b281f39f.jpg']
